# DSCI 519 · Homework 4
## Customer Entity Resolution and an AI-Written Log Parser

**Covers:** Session 4 (Chapter 7). **Points:** 100. **Submit:** notebook with outputs, the PDF made by the last cell,
`hw4_clusters.csv`, and `hw4_parsed_log.csv`.

### The situation
**Bluestem Coffee Co.** (fictional) runs a web store and a separate loyalty program. Marketing wants one view of each
customer: who buys online, who is a loyalty member, and who is both. The two systems were never linked, people use
nicknames and work emails, staff mistype names, and family members share addresses and phones.

You also received the point-of-sale register log from Valentine's Day morning, written by three different systems.

### Data
* `bluestem_web_customers.csv` (83 records), `bluestem_loyalty_members.csv` (86 records)
* `bluestem_dev_pairs.csv`: 40 labeled record pairs (`is_match` 1 or 0), including some deliberately hard non-matches
* `pos_register.log` (60 lines) and `pos_log_dev_gold.csv` (expected parse of the first 15 lines)

### Rubric
| Part | What | Points |
|---|---|---|
| A | Entity resolution pipeline: standardize, block, score, LLM for the gray zone, cluster, evaluate | 45 |
| B | Golden customer records with documented survivorship rules | 15 |
| C | LLM-written log parser, tested and iterated | 25 |
| D | Deliverables and reflection (cost, error trade-offs) | 15 |

Part A is partly scored on hidden labels (10 of its 45 points): pairwise F1 of your `hw4_clusters.csv` over all record
pairs (F1 of 0.90 or higher earns full marks for that component). Session 4, 0.6 and 3.3 show how pairwise F1 of
clusters is computed.

In [1]:
#@title Setup: install packages and the course helper (run me first)
# Installs the vendor SDKs (quietly) and writes the course helper module llmkit.py
# into the Colab working folder. Run this cell once per Colab session.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "anthropic", "google-genai", "pydantic", "rapidfuzz"], check=False)
LLMKIT_SOURCE = r'''"""
llmkit: a small, vendor-agnostic LLM client for DSCI 519 (Advanced Business Analytics).

Why this file exists
--------------------
Every AI vendor ships its own Python SDK, and each SDK shapes requests and
responses a little differently. OpenAI returns `choices[0].message.content`,
Anthropic returns a list of `content` blocks, Google returns `.text`, and the
token counters all have different names. If we wrote our pipelines directly
against one SDK, switching vendors would mean rewriting every pipeline.

This module is an *adapter*: one small interface (`LLM.chat`, `LLM.ask`,
`LLM.extract`) on top of several providers. Your pipeline code talks to the
adapter; the adapter talks to the vendor. That is the same design idea that
data engineers use for database drivers (one SQL interface, many databases).

Supported providers
-------------------
    "openai"      OpenAI API                 (key: OPENAI_API_KEY)
    "anthropic"   Anthropic Claude API       (key: ANTHROPIC_API_KEY)
    "google"      Google Gemini API          (key: GOOGLE_API_KEY or GEMINI_API_KEY)
    "groq"        Groq (OpenAI-compatible)   (key: GROQ_API_KEY)        free tier
    "openrouter"  OpenRouter (OpenAI-compat) (key: OPENROUTER_API_KEY)  some free models
    "mistral"     Mistral (OpenAI-compat)    (key: MISTRAL_API_KEY)
    "ollama"      Local models via Ollama    (no key; needs a local server)
    "compatible"  Any other OpenAI-compatible endpoint (pass base_url and key_name)
    "mock"        Offline fake model for testing pipeline wiring (no key, no cost)

Model names change often. The defaults below were checked in October 2026.
If a call fails with "model not found", run `LLM(provider).list_models()`
and pick a current model from your provider's list.

Quick start
-----------
    from llmkit import LLM
    llm = LLM("groq")                              # or "openai", "anthropic", "google", ...
    print(llm.ask("Say hello in five words."))

    from pydantic import BaseModel
    class Sentiment(BaseModel):
        label: str
        score: float
    result = llm.extract("I love this product!", Sentiment,
                         instructions="Classify the sentiment of the review.")
    print(result.label, result.score)

    llm.usage()                                     # calls, tokens, latency, cost
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Type

__version__ = "1.1.0"

# ---------------------------------------------------------------------------
# 1. Provider registry: the only place where vendor-specific settings live.
# ---------------------------------------------------------------------------
PROVIDERS: Dict[str, Dict[str, Any]] = {
    "openai": {
        "style": "openai",
        "key_names": ["OPENAI_API_KEY"],
        "base_url": None,
        "default_model": "gpt-6-luna",
    },
    "anthropic": {
        "style": "anthropic",
        "key_names": ["ANTHROPIC_API_KEY"],
        "base_url": None,
        "default_model": "claude-haiku-4-5",
    },
    "google": {
        "style": "google",
        "key_names": ["GOOGLE_API_KEY", "GEMINI_API_KEY"],
        "base_url": None,
        "default_model": "gemini-3.5-flash-lite",
    },
    "groq": {
        "style": "compatible",
        "key_names": ["GROQ_API_KEY"],
        "base_url": "https://api.groq.com/openai/v1",
        "default_model": "llama-3.3-70b-versatile",
    },
    "openrouter": {
        "style": "compatible",
        "key_names": ["OPENROUTER_API_KEY"],
        "base_url": "https://openrouter.ai/api/v1",
        "default_model": "openai/gpt-oss-20b:free",
    },
    "mistral": {
        "style": "compatible",
        "key_names": ["MISTRAL_API_KEY"],
        "base_url": "https://api.mistral.ai/v1",
        "default_model": "mistral-small-latest",
    },
    "ollama": {
        "style": "compatible",
        "key_names": [],
        "base_url": "http://localhost:11434/v1",
        "default_model": "llama3.2",
    },
    "compatible": {
        "style": "compatible",
        "key_names": [],
        "base_url": None,
        "default_model": None,
    },
    "mock": {
        "style": "mock",
        "key_names": [],
        "base_url": None,
        "default_model": "mock-1",
    },
}


# ---------------------------------------------------------------------------
# 2. Secrets: never paste API keys into notebook cells.
# ---------------------------------------------------------------------------
def get_secret(name: str, prompt_if_missing: bool = True) -> Optional[str]:
    """Look up a secret in (1) Colab Secrets, (2) environment variables,
    (3) an interactive hidden prompt. Returns None if nothing is found."""
    # (1) Google Colab Secrets panel (the key icon in the left sidebar)
    try:
        from google.colab import userdata  # type: ignore

        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass  # secret not defined or notebook access not granted
    except ImportError:
        pass  # not running in Colab
    # (2) Environment variable (local Jupyter, .env files, CI)
    value = os.environ.get(name)
    if value:
        return value
    # (3) Ask the user without echoing the key on screen
    if prompt_if_missing:
        try:
            from getpass import getpass

            value = getpass(f"Paste your {name} (input is hidden): ").strip()
            if value:
                os.environ[name] = value  # remember for this session only
                return value
        except Exception:
            pass
    return None


# ---------------------------------------------------------------------------
# 3. A normalized response object: the same shape for every vendor.
# ---------------------------------------------------------------------------
@dataclass
class LLMResponse:
    text: str
    provider: str
    model: str
    input_tokens: int = 0
    output_tokens: int = 0
    latency_s: float = 0.0
    finish_reason: Optional[str] = None
    cached: bool = False
    raw: Any = field(default=None, repr=False)

    @property
    def total_tokens(self) -> int:
        return (self.input_tokens or 0) + (self.output_tokens or 0)


class LLMError(RuntimeError):
    """Raised when a call cannot be completed (after retries)."""


class ExtractionError(LLMError):
    """Raised when the model's output never matched the schema."""

    def __init__(self, message: str, last_text: str = "", errors: Optional[List[str]] = None):
        super().__init__(message)
        self.last_text = last_text
        self.errors = errors or []


# ---------------------------------------------------------------------------
# 4. JSON helpers used for structured outputs.
# ---------------------------------------------------------------------------
_FENCE_RE = re.compile(r"^```(?:json|JSON)?\s*|\s*```$", re.MULTILINE)


def parse_json_object(text: str) -> Any:
    """Turn a model reply into a Python object. Tolerates markdown fences and
    short prose before or after the JSON (a common model habit)."""
    if text is None:
        raise ValueError("empty reply")
    cleaned = _FENCE_RE.sub("", text.strip()).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    # Fall back to the outermost {...} or [...] span.
    for open_ch, close_ch in (("{", "}"), ("[", "]")):
        start, end = cleaned.find(open_ch), cleaned.rfind(close_ch)
        if start != -1 and end > start:
            try:
                return json.loads(cleaned[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError("reply is not valid JSON")


def schema_prompt(schema: Type[Any]) -> str:
    """The text we append to instructions so ANY model knows the exact shape
    we expect. Works with every vendor because it is just text."""
    schema_json = json.dumps(schema.model_json_schema(), indent=2)
    return (
        "Return ONLY one JSON object (no prose, no markdown code fences) that "
        "conforms to this JSON Schema:\n"
        f"{schema_json}\n"
        "Rules: use null when an optional value is not present in the input; "
        "never invent values; do not add keys that are not in the schema."
    )


def estimate_tokens(text: str) -> int:
    """Vendor-neutral rule of thumb: about 4 characters per token for English.
    Every vendor uses a different tokenizer, so treat this as an estimate."""
    return max(1, round(len(text or "") / 4))


# ---------------------------------------------------------------------------
# 5. The client.
# ---------------------------------------------------------------------------
class LLM:
    """One interface, many vendors.

    Parameters
    ----------
    provider : str            one of PROVIDERS (e.g., "groq", "openai", "google")
    model : str               model id; defaults to the provider's default_model
    api_key : str             optional; otherwise read with get_secret()
    base_url : str            only for "compatible"/"ollama" style endpoints
    key_name : str            secret name to read for a "compatible" provider
    temperature : float       default sampling temperature (None = provider default)
    max_tokens : int          default cap on output tokens
    min_interval : float      minimum seconds between calls (free-tier rate limits)
    max_retries : int         retries for rate limits, timeouts, server errors
    cache : bool              reuse identical past responses (saves money on reruns)
    cache_path : str          where cached responses are stored (JSON lines)
    price_per_million : tuple (input_usd, output_usd) per 1M tokens, for cost reports
    verbose : bool            print one line per call
    reasoning_headroom : int  if a reply is cut off by a small max_tokens (reasoning models
                              spend tokens on hidden "thinking"), retry once with this many
                              tokens and use it as the minimum from then on (0 = never)
    """

    def __init__(
        self,
        provider: str = "groq",
        model: Optional[str] = None,
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
        key_name: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: int = 2048,
        min_interval: float = 0.0,
        max_retries: int = 4,
        timeout: float = 90.0,
        cache: bool = False,
        cache_path: str = "llm_cache.jsonl",
        price_per_million: Optional[tuple] = None,
        verbose: bool = False,
        reasoning_headroom: int = 2048,
    ):
        provider = provider.lower().strip()
        if provider not in PROVIDERS:
            raise ValueError(f"Unknown provider '{provider}'. Choose one of: {sorted(PROVIDERS)}")
        cfg = PROVIDERS[provider]
        self.provider = provider
        self.style = cfg["style"]
        self.model = model or cfg["default_model"]
        if not self.model:
            raise ValueError("Please pass model=... for this provider.")
        self.base_url = base_url or cfg["base_url"]
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.min_interval = min_interval
        self.max_retries = max_retries
        self.timeout = timeout
        self.price_per_million = price_per_million
        self.verbose = verbose
        self.reasoning_headroom = reasoning_headroom
        self._token_floor = 0  # raised automatically if the model needs room to "think"
        self.calls: List[Dict[str, Any]] = []  # one row per API call (for usage reports)
        self.warnings: List[str] = []
        self.last: Optional[LLMResponse] = None
        self._last_call_time = 0.0
        self._dropped_params: set = set()  # params a model rejected (e.g., temperature)
        self._mock_schema = None

        # API key (not needed for mock or local ollama)
        names = ([key_name] if key_name else []) + list(cfg["key_names"])
        self._api_key = api_key
        if self._api_key is None and self.style != "mock" and names:
            for i, name in enumerate(names):
                self._api_key = get_secret(name, prompt_if_missing=(i == len(names) - 1))
                if self._api_key:
                    break
            if not self._api_key:
                raise LLMError(
                    f"No API key found for '{provider}'. Add {names[0]} to Colab Secrets "
                    "(key icon in the left sidebar) and enable notebook access."
                )
        if provider == "ollama" and not self._api_key:
            self._api_key = "ollama"  # Ollama ignores the key but the SDK wants one

        # Response cache (optional)
        self.cache_enabled = cache
        self.cache_path = cache_path
        self._cache: Dict[str, Dict[str, Any]] = {}
        if cache and os.path.exists(cache_path):
            with open(cache_path, "r", encoding="utf-8") as fh:
                for line in fh:
                    try:
                        row = json.loads(line)
                        self._cache[row["key"]] = row
                    except Exception:
                        continue

        self._client = None  # created lazily on the first call

    def __repr__(self) -> str:
        return f"LLM(provider='{self.provider}', model='{self.model}')"

    # ---------------------------------------------------------------- clients
    def _get_client(self):
        if self._client is not None:
            return self._client
        if self.style in ("openai", "compatible"):
            from openai import OpenAI

            kwargs = {"api_key": self._api_key, "timeout": self.timeout, "max_retries": 0}
            if self.base_url:
                kwargs["base_url"] = self.base_url
            self._client = OpenAI(**kwargs)
        elif self.style == "anthropic":
            import anthropic

            kwargs = {"api_key": self._api_key, "timeout": self.timeout, "max_retries": 0}
            if self.base_url:
                kwargs["base_url"] = self.base_url
            self._client = anthropic.Anthropic(**kwargs)
        elif self.style == "google":
            from google import genai

            kwargs = {"api_key": self._api_key}
            if self.base_url:
                from google.genai import types

                kwargs["http_options"] = types.HttpOptions(base_url=self.base_url)
            self._client = genai.Client(**kwargs)
        else:
            self._client = "mock"
        return self._client

    # ------------------------------------------------------- vendor adapters
    def _call_openai_style(self, system, messages, temperature, max_tokens, json_mode):
        client = self._get_client()
        msgs = ([{"role": "system", "content": system}] if system else []) + messages
        kwargs: Dict[str, Any] = {"model": self.model, "messages": msgs}
        # OpenAI's newer models use max_completion_tokens; most compatible
        # servers still use max_tokens.
        if self.style == "openai":
            kwargs["max_completion_tokens"] = max_tokens
        else:
            kwargs["max_tokens"] = max_tokens
        if temperature is not None and "temperature" not in self._dropped_params:
            kwargs["temperature"] = temperature
        if json_mode and "response_format" not in self._dropped_params:
            kwargs["response_format"] = {"type": "json_object"}
        resp = client.chat.completions.create(**kwargs)
        choice = resp.choices[0]
        usage = getattr(resp, "usage", None)
        return LLMResponse(
            text=(choice.message.content or ""),
            provider=self.provider,
            model=getattr(resp, "model", self.model) or self.model,
            input_tokens=getattr(usage, "prompt_tokens", 0) or 0,
            output_tokens=getattr(usage, "completion_tokens", 0) or 0,
            finish_reason=getattr(choice, "finish_reason", None),
            raw=resp,
        )

    def _call_anthropic(self, system, messages, temperature, max_tokens, json_mode):
        client = self._get_client()
        kwargs: Dict[str, Any] = {"model": self.model, "max_tokens": max_tokens, "messages": messages}
        if system:
            kwargs["system"] = system
        if temperature is not None and "temperature" not in self._dropped_params:
            kwargs["extra_body"] = {"temperature": temperature}
        # Anthropic has no "JSON mode" flag; the schema prompt does the work.
        resp = client.messages.create(**kwargs)
        text = "".join(getattr(b, "text", "") for b in resp.content if getattr(b, "type", "") == "text")
        usage = getattr(resp, "usage", None)
        return LLMResponse(
            text=text,
            provider=self.provider,
            model=getattr(resp, "model", self.model) or self.model,
            input_tokens=getattr(usage, "input_tokens", 0) or 0,
            output_tokens=getattr(usage, "output_tokens", 0) or 0,
            finish_reason=getattr(resp, "stop_reason", None),
            raw=resp,
        )

    def _call_google(self, system, messages, temperature, max_tokens, json_mode):
        from google.genai import types

        client = self._get_client()
        contents = [
            types.Content(
                role=("model" if m["role"] == "assistant" else "user"),
                parts=[types.Part(text=m["content"])],
            )
            for m in messages
        ]
        cfg: Dict[str, Any] = {
            "max_output_tokens": max_tokens,
            # we never pass tools, so switch off automatic function calling
            "automatic_function_calling": types.AutomaticFunctionCallingConfig(disable=True),
        }
        if system:
            cfg["system_instruction"] = system
        if temperature is not None and "temperature" not in self._dropped_params:
            cfg["temperature"] = temperature
        if json_mode and "response_format" not in self._dropped_params:
            cfg["response_mime_type"] = "application/json"
        resp = client.models.generate_content(
            model=self.model, contents=contents, config=types.GenerateContentConfig(**cfg)
        )
        usage = getattr(resp, "usage_metadata", None)
        finish = None
        try:
            finish = str(resp.candidates[0].finish_reason)
        except Exception:
            pass
        return LLMResponse(
            text=(resp.text or ""),
            provider=self.provider,
            model=self.model,
            input_tokens=getattr(usage, "prompt_token_count", 0) or 0,
            output_tokens=(getattr(usage, "candidates_token_count", 0) or 0)
            + (getattr(usage, "thoughts_token_count", 0) or 0),
            finish_reason=finish,
            raw=resp,
        )

    def _call_mock(self, system, messages, temperature, max_tokens, json_mode):
        """A fake model. It never 'understands' anything: it returns a fixed
        reply, or (for structured calls) the simplest object that fits the
        schema. Use it to test that your pipeline code runs end to end."""
        prompt_text = (system or "") + " ".join(m["content"] for m in messages)
        if json_mode and self._mock_schema is not None:
            text = json.dumps(_mock_instance(self._mock_schema.model_json_schema()))
        elif json_mode:
            text = "{}"
        else:
            text = "MOCK RESPONSE (provider='mock' returns placeholder text)."
        return LLMResponse(
            text=text,
            provider="mock",
            model=self.model,
            input_tokens=estimate_tokens(prompt_text),
            output_tokens=estimate_tokens(text),
            finish_reason="stop",
        )

    # --------------------------------------------------------- error policy
    @staticmethod
    def _status_code(exc: Exception) -> Optional[int]:
        for attr in ("status_code", "code", "status"):
            val = getattr(exc, attr, None)
            if isinstance(val, int):
                return val
        resp = getattr(exc, "response", None)
        val = getattr(resp, "status_code", None)
        return val if isinstance(val, int) else None

    def _is_retryable(self, exc: Exception) -> bool:
        code = self._status_code(exc)
        name = type(exc).__name__.lower()
        msg = str(exc).lower()
        if code in (408, 409, 425, 429, 500, 502, 503, 504, 529):
            return True
        if any(w in name for w in ("ratelimit", "timeout", "connection", "overloaded", "unavailable")):
            return True
        return "resource_exhausted" in msg or "rate limit" in msg or "overloaded" in msg

    def _unsupported_param(self, exc: Exception) -> Optional[str]:
        """Some models reject parameters (e.g., reasoning models reject a custom
        temperature). Detect that so we can retry without the parameter."""
        code = self._status_code(exc)
        msg = str(exc).lower()
        if code not in (400, 422, None):
            return None
        for param in ("temperature", "response_format", "response_mime_type"):
            if param in msg and ("support" in msg or "unrecognized" in msg or "invalid" in msg or "not allowed" in msg):
                return "response_format" if param == "response_mime_type" else param
        return None

    # ------------------------------------------------------------- the call
    def chat(
        self,
        messages: List[Dict[str, str]],
        system: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        json_mode: bool = False,
    ) -> LLMResponse:
        """Send a conversation and get one normalized LLMResponse back.

        messages: [{"role": "user" | "assistant", "content": "..."}, ...]
        system:   standing instructions (kept separate from messages because
                  vendors place them differently)
        """
        temperature = self.temperature if temperature is None else temperature
        max_tokens = max(max_tokens or self.max_tokens, self._token_floor)
        for m in messages:
            if m.get("role") not in ("user", "assistant"):
                raise ValueError("messages may only use roles 'user' and 'assistant'; pass system=... separately")

        # 1) Cache lookup: identical request -> identical (free) answer.
        key = None
        if self.cache_enabled:
            key = hashlib.sha256(
                json.dumps(
                    [self.provider, self.model, system, messages, temperature, max_tokens, json_mode],
                    sort_keys=True,
                ).encode("utf-8")
            ).hexdigest()
            if key in self._cache:
                row = self._cache[key]
                resp = LLMResponse(
                    text=row["text"],
                    provider=self.provider,
                    model=self.model,
                    input_tokens=row.get("input_tokens", 0),
                    output_tokens=row.get("output_tokens", 0),
                    cached=True,
                )
                self._log(resp)
                self.last = resp
                return resp

        adapter = {
            "openai": self._call_openai_style,
            "compatible": self._call_openai_style,
            "anthropic": self._call_anthropic,
            "google": self._call_google,
            "mock": self._call_mock,
        }[self.style]

        # 2) Call with polite pacing, retries, and parameter fallback.
        attempt = 0
        while True:
            wait = self.min_interval - (time.time() - self._last_call_time)
            if wait > 0:
                time.sleep(wait)
            start = time.time()
            self._last_call_time = start
            try:
                resp = adapter(system, messages, temperature, max_tokens, json_mode)
                resp.latency_s = round(time.time() - start, 3)
                break
            except Exception as exc:  # vendor SDKs raise different classes
                bad_param = self._unsupported_param(exc)
                if bad_param and bad_param not in self._dropped_params:
                    self._dropped_params.add(bad_param)
                    self._warn(f"Model '{self.model}' rejected '{bad_param}'; retrying without it.")
                    continue
                attempt += 1
                if attempt > self.max_retries or not self._is_retryable(exc):
                    raise LLMError(f"{self.provider} call failed: {type(exc).__name__}: {exc}") from exc
                delay = min(60.0, 2.0 ** attempt)
                self._warn(f"{type(exc).__name__} (attempt {attempt}); waiting {delay:.0f}s before retrying.")
                time.sleep(delay)

        cut_off = str(resp.finish_reason).lower() in ("length", "max_tokens", "finishreason.max_tokens")
        if (cut_off and (json_mode or not resp.text.strip()) and self.style != "mock"
                and self.reasoning_headroom and max_tokens < self.reasoning_headroom):
            # Typical of reasoning models: the hidden "thinking" used up the budget before the
            # answer was written. Retry once with more room and keep that minimum for later calls.
            self._log(resp)
            self._token_floor = self.reasoning_headroom
            self._warn(
                f"Reply cut off at max_tokens={max_tokens} (this model spends tokens on hidden reasoning). "
                f"Retrying with max_tokens={self.reasoning_headroom}; later calls from this client use at least that."
            )
            return self.chat(messages, system=system, temperature=temperature,
                             max_tokens=self.reasoning_headroom, json_mode=json_mode)
        if not resp.text.strip() and cut_off:
            self._warn(
                "The reply is empty because the token limit was reached (reasoning models spend "
                "tokens 'thinking'). Increase max_tokens."
            )

        # 3) Save to cache and usage log.
        if self.cache_enabled and key:
            row = {"key": key, "text": resp.text, "input_tokens": resp.input_tokens, "output_tokens": resp.output_tokens}
            self._cache[key] = row
            with open(self.cache_path, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(row) + "\n")
        self._log(resp)
        self.last = resp
        return resp

    def ask(self, prompt: str, system: Optional[str] = None, **kwargs) -> str:
        """Single-turn convenience wrapper: prompt in, text out."""
        return self.chat([{"role": "user", "content": prompt}], system=system, **kwargs).text

    def extract(
        self,
        content: Any,
        schema: Type[Any],
        instructions: Optional[str] = None,
        max_repairs: int = 2,
        **kwargs,
    ):
        """Structured output that works with every vendor.

        1. Put the JSON Schema of `schema` (a Pydantic model) into the system prompt.
        2. Ask for JSON (using the vendor's JSON mode when it has one).
        3. Parse and validate the reply with Pydantic.
        4. If validation fails, show the model its own error and ask again
           (a "repair loop"), up to `max_repairs` times.
        Returns an instance of `schema`, or raises ExtractionError.
        """
        system = ((instructions.strip() + "\n\n") if instructions else "") + schema_prompt(schema)
        if not isinstance(content, str):
            content = json.dumps(content, default=str, ensure_ascii=False)
        messages = [{"role": "user", "content": content}]
        errors: List[str] = []
        text = ""
        self._mock_schema = schema
        try:
            for attempt in range(max_repairs + 1):
                resp = self.chat(messages, system=system, json_mode=True, **kwargs)
                text = resp.text
                try:
                    data = parse_json_object(text)
                    obj = schema.model_validate(data)
                    resp.attempts = attempt + 1  # type: ignore[attr-defined]
                    return obj
                except Exception as exc:  # JSON error or pydantic ValidationError
                    err = _short_error(exc)
                    errors.append(err)
                    messages = messages + [
                        {"role": "assistant", "content": text or "(empty reply)"},
                        {
                            "role": "user",
                            "content": "Your previous reply could not be used because: "
                            f"{err}\nReply again with ONLY the corrected JSON object.",
                        },
                    ]
        finally:
            self._mock_schema = None
        raise ExtractionError(
            f"No schema-valid reply after {max_repairs + 1} attempts. Last error: {errors[-1]}",
            last_text=text,
            errors=errors,
        )

    # ---------------------------------------------------------- bookkeeping
    def _warn(self, msg: str) -> None:
        self.warnings.append(msg)
        print(f"[llmkit] {msg}")

    def _cost(self, resp: LLMResponse) -> Optional[float]:
        if not self.price_per_million or resp.cached:
            return 0.0 if resp.cached else None
        p_in, p_out = self.price_per_million
        return (resp.input_tokens * p_in + resp.output_tokens * p_out) / 1_000_000

    def _log(self, resp: LLMResponse) -> None:
        row = {
            "time": time.strftime("%H:%M:%S"),
            "provider": resp.provider,
            "model": resp.model,
            "input_tokens": resp.input_tokens,
            "output_tokens": resp.output_tokens,
            "latency_s": resp.latency_s,
            "cached": resp.cached,
            "cost_usd": self._cost(resp),
        }
        self.calls.append(row)
        if self.verbose:
            print(
                f"[llmkit] {row['provider']}/{row['model']} in={row['input_tokens']} "
                f"out={row['output_tokens']} {row['latency_s']}s{' (cached)' if resp.cached else ''}"
            )

    def usage(self):
        """Summary of all calls made by this client (a pandas DataFrame if available)."""
        try:
            import pandas as pd

            df = pd.DataFrame(self.calls)
            if df.empty:
                print("No calls yet.")
                return df
            summary = {
                "calls": len(df),
                "cached_calls": int(df["cached"].sum()),
                "input_tokens": int(df["input_tokens"].sum()),
                "output_tokens": int(df["output_tokens"].sum()),
                "avg_latency_s": round(df.loc[~df["cached"], "latency_s"].mean(), 2) if (~df["cached"]).any() else 0.0,
                "est_cost_usd": (round(df["cost_usd"].sum(), 6) if df["cost_usd"].notna().any() else "set price_per_million"),
            }
            return pd.Series(summary, name=f"{self.provider}/{self.model}", dtype=object)
        except ImportError:
            return self.calls

    def reset_usage(self) -> None:
        self.calls = []

    def list_models(self) -> List[str]:
        """Ask the provider which models your key can use."""
        client = self._get_client()
        if self.style in ("openai", "compatible"):
            return sorted(m.id for m in client.models.list())
        if self.style == "anthropic":
            return sorted(m.id for m in client.models.list())
        if self.style == "google":
            return sorted(m.name.replace("models/", "") for m in client.models.list())
        return ["mock-1"]


# ---------------------------------------------------------------------------
# 6. Small private helpers.
# ---------------------------------------------------------------------------
def _short_error(exc: Exception) -> str:
    """Compact error text that a model (and a student) can read."""
    try:
        from pydantic import ValidationError

        if isinstance(exc, ValidationError):
            parts = []
            for e in exc.errors()[:6]:
                loc = ".".join(str(x) for x in e.get("loc", ())) or "(root)"
                parts.append(f"{loc}: {e.get('msg')}")
            return "; ".join(parts)
    except ImportError:
        pass
    return f"{type(exc).__name__}: {exc}"[:400]


def _mock_instance(schema: Dict[str, Any], defs: Optional[Dict[str, Any]] = None) -> Any:
    """Build the simplest value that satisfies a JSON Schema (used by provider='mock')."""
    defs = defs if defs is not None else schema.get("$defs", {})
    if "$ref" in schema:
        return _mock_instance(defs.get(schema["$ref"].split("/")[-1], {}), defs)
    if "enum" in schema:
        return schema["enum"][0]
    if "const" in schema:
        return schema["const"]
    for key in ("anyOf", "oneOf", "allOf"):
        if key in schema:
            options = [s for s in schema[key] if s.get("type") != "null"] or schema[key]
            return _mock_instance(options[0], defs)
    if "default" in schema and schema["default"] is not None:
        return schema["default"]
    typ = schema.get("type")
    if typ == "object" or "properties" in schema:
        props = schema.get("properties", {})
        return {k: _mock_instance(v, defs) for k, v in props.items()}
    if typ == "array":
        return []
    if typ == "integer":
        return int(schema.get("minimum", 0))
    if typ == "number":
        return float(schema.get("minimum", 0.0))
    if typ == "boolean":
        return False
    if typ == "null":
        return None
    return "mock"
'''
with open("llmkit.py", "w", encoding="utf-8") as fh:
    fh.write(LLMKIT_SOURCE)
import importlib, llmkit
importlib.reload(llmkit)
print("llmkit", llmkit.__version__, "is ready.")

llmkit 1.1.0 is ready.


### Choose your AI provider

Everything in this course works with any of the providers below. Pick one, make sure its API key is stored in
**Colab Secrets** (click the key icon in the left sidebar, add the secret name shown, and switch on
*Notebook access*), and run the cell. You never paste a key into the notebook itself.

| `PROVIDER` | Secret name | Cost notes (October 2026, verify before use) | Suggested `MIN_INTERVAL` |
|---|---|---|---|
| `"groq"` | `GROQ_API_KEY` | Free tier, no credit card, generous daily limits | 2 |
| `"google"` | `GOOGLE_API_KEY` | Free tier for Flash models (low daily quota; your data may be used for training) | 6 |
| `"openai"` | `OPENAI_API_KEY` | Paid, prepaid credit; small models cost cents for this course | 0 |
| `"anthropic"` | `ANTHROPIC_API_KEY` | Paid, prepaid credit; Haiku is the low-cost tier | 0 |
| `"openrouter"` | `OPENROUTER_API_KEY` | Many models behind one key; some `:free` models | 3 |
| `"mistral"` | `MISTRAL_API_KEY` | Free experimentation tier available | 2 |
| `"mock"` | none | Offline fake model: returns placeholders, useful only to test that code runs | 0 |

`MODEL = None` uses the provider's default from `llmkit.PROVIDERS`. Model names change frequently; if you get a
"model not found" error, run `llm.list_models()` and copy a current name into `MODEL`.

In [2]:
from llmkit import LLM, get_secret, estimate_tokens

PROVIDER = "google"      # "groq" | "google" | "openai" | "anthropic" | "openrouter" | "mistral" | "mock"
MODEL = None           # None = provider default, or a specific model id string
MIN_INTERVAL = 2.0     # seconds between calls (protects free-tier rate limits)

llm = LLM(PROVIDER, model=MODEL, min_interval=MIN_INTERVAL, cache=True)
print(llm)

Paste your GEMINI_API_KEY (input is hidden): ··········
LLM(provider='google', model='gemini-3.5-flash-lite')


In [3]:
#@title Get the data files
# Course data: this cell makes the homework files available in /content/hw4.
# Option 1 (recommended): your instructor sets COURSE_DATA_URL to where the course data folder is hosted.
# Option 2: upload the files when prompted (download them from the course site first).
import os, urllib.request
COURSE_DATA_URL = ""   # e.g. "https://raw.githubusercontent.com/<account>/<repo>/main/data"
FOLDER = "hw4"
FILES = ['bluestem_web_customers.csv', 'bluestem_loyalty_members.csv', 'bluestem_dev_pairs.csv', 'pos_register.log', 'pos_log_dev_gold.csv']
os.makedirs(FOLDER, exist_ok=True)
missing = [f for f in FILES if not os.path.exists(os.path.join(FOLDER, f))]
for f in list(missing):
    if COURSE_DATA_URL:
        try:
            urllib.request.urlretrieve(f"{COURSE_DATA_URL}/{FOLDER}/{f}", os.path.join(FOLDER, f))
            missing.remove(f)
        except Exception as exc:
            print("Download failed for", f, exc)
if missing:
    try:
        from google.colab import files as colab_files
        print("Please upload:", missing)
        uploaded = colab_files.upload()
        for name, content in uploaded.items():
            with open(os.path.join(FOLDER, name), "wb") as fh:
                fh.write(content)
    except ImportError:
        print("Not in Colab: copy these files into", FOLDER, ":", missing)
print("Files present:", sorted(os.listdir(FOLDER)))

Please upload: ['bluestem_web_customers.csv', 'bluestem_loyalty_members.csv', 'bluestem_dev_pairs.csv', 'pos_register.log', 'pos_log_dev_gold.csv']


Saving bluestem_dev_pairs.csv to bluestem_dev_pairs.csv
Saving bluestem_loyalty_members.csv to bluestem_loyalty_members.csv
Saving bluestem_web_customers.csv to bluestem_web_customers.csv
Saving pos_log_dev_gold.csv to pos_log_dev_gold.csv
Saving pos_register.log to pos_register.log
Files present: ['bluestem_dev_pairs.csv', 'bluestem_loyalty_members.csv', 'bluestem_web_customers.csv', 'pos_log_dev_gold.csv', 'pos_register.log']


In [4]:
import pandas as pd, numpy as np, re, json
web = pd.read_csv("hw4/bluestem_web_customers.csv", dtype=str).fillna("")
loyal = pd.read_csv("hw4/bluestem_loyalty_members.csv", dtype=str).fillna("")
dev_pairs = pd.read_csv("hw4/bluestem_dev_pairs.csv")
log_lines = open("hw4/pos_register.log").read().splitlines()
log_dev = pd.read_csv("hw4/pos_log_dev_gold.csv", dtype=str).fillna("")
log_dev["line_no"] = log_dev.line_no.astype(int)
display(web.head(3), loyal.head(3), dev_pairs.head())

,record_id,first_name,last_name,email,phone,street_address,city,state,zip,created_at
0,W1000,Katherine,Nugyen,katherine.nguyen59@gmail.com,+16054344236,4300 S Western Ave,Brandon,SD,57007,2024-10-28
1,W1001,Leila,Peterosn,leila.peterson9@gmail.com,(605) 600-6820,7820 S Cliff Ave #27,Sioux Falls,SD,57196,2023-02-25
2,W1002,Alex,Walsh,alexander.walsh22@gmail.com,+16058822206,6160 S Lake Ave,Harrisburg,SD,57091,2023-04-22


,record_id,member_name,email,mobile,home_address,birth_year,points_balance,member_since
0,L2000,Patricia Moreno,patricia.moreno@yahoo.com,,"6891 S Western Ave, Brandon, SD 57067",1994,1610,2021-11-18
1,L2001,"WALSH, William",william.walsh74@hotmail.com,(605) 872-1289,"6056 S Main Ave, Harrisburg, SD 57026",1994,1090,2024-10-01
2,L2002,Leila Peterson,leila.peterson@avera.org,605-600-6820,"7820 S Cliff Ave Apt 27, Sioux Falls, SD 57196",,3140,2023-09-16


,record_id_a,record_id_b,is_match
0,W1038,L2015,1
1,W1019,L2028,1
2,W1009,L2031,1
3,W1025,W1065,1
4,L2044,L2060,0


---
# Part A. Entity resolution (45 points)

### A1. Standardize both sources into one table (10)
Build `people` with one row per source record and columns `record_id, first, last, email, phone, street, city, zip,
birth_year`, plus a `first_canon` column that maps nicknames to formal names (Bill to William, Kate to Katherine, ...).
Loyalty names come as "First Last" or "LAST, First"; loyalty addresses are one string ("street, city, ST zip").

*Tip:* a nickname dictionary is a perfect job for a coding companion ("give me a Python dict of common English nicknames
to formal first names"), but **review it**: models add questionable entries.

*Where this was taught:* Session 4, 0.3 (flipping "LAST, First", nickname and abbreviation dicts, phone digits) and
0.1 (`.str.extract` with named groups, which splits a one-string address into columns).

In [5]:
# A1
# Nickname dictionary (reviewed: only nicknames that clearly map to one formal name)
NICKNAMES = {
    "alex": "alexander", "ben": "benjamin", "benny": "benjamin",
    "bill": "william", "billy": "william", "will": "william", "willie": "william",
    "bob": "robert", "bobby": "robert", "rob": "robert", "robbie": "robert",
    "kate": "katherine", "kathy": "katherine", "katie": "katherine", "kat": "katherine",
    "jo": "josephine", "josie": "josephine",
    "jon": "jonathan", "jonny": "jonathan", "johnny": "jonathan",
    "mike": "michael", "mikey": "michael", "mick": "michael",
    "meg": "margaret", "maggie": "margaret", "peggy": "margaret", "peg": "margaret",
    "becky": "rebecca", "becca": "rebecca",
    "pat": "patricia", "patty": "patricia", "trish": "patricia", "tricia": "patricia",
    "vicky": "victoria", "vicki": "victoria", "tori": "victoria",
    "liz": "elizabeth", "beth": "elizabeth", "eliza": "elizabeth", "lizzie": "elizabeth",
    "gabe": "gabriel", "tom": "thomas", "tommy": "thomas",
    "dan": "daniel", "danny": "daniel", "nick": "nicholas",
    "tony": "anthony", "jen": "jennifer", "jenny": "jennifer",
    "chris": "christopher", "sam": "samuel",
}

# Street abbreviations so "South Main Avenue" and "S Main Ave" match
STREET_ABBR = {"SOUTH": "S", "NORTH": "N", "EAST": "E", "WEST": "W",
               "AVENUE": "AVE", "STREET": "ST", "ROAD": "RD", "DRIVE": "DR", "APARTMENT": "APT"}

# Cleaning helpers
def clean_name(s):
    return re.sub(r"[^a-z ]", "", s.lower()).strip()

def clean_email(s):
    return s.strip().lower()

def clean_phone(s):
    digits = re.sub(r"\D", "", s)
    return digits[-10:] if len(digits) >= 10 else ""

def clean_street(s):
    s = s.upper().replace("#", " APT ").replace(".", "")
    return " ".join(STREET_ABBR.get(w, w) for w in s.split())

# Flip "LAST, First" to first/last; otherwise split "First Last"
def split_member_name(s):
    if "," in s:
        last, first = [p.strip() for p in s.split(",", 1)]
    else:
        first, last = s.strip().split(" ", 1)
    return pd.Series({"first": first, "last": last})

# Web records: columns are already separate
web_std = pd.DataFrame({
    "record_id": web.record_id,
    "first": web.first_name.map(clean_name),
    "last": web.last_name.map(clean_name),
    "email": web.email.map(clean_email),
    "phone": web.phone.map(clean_phone),
    "street": web.street_address.map(clean_street),
    "city": web.city.str.strip().str.title(),
    "zip": web.zip.str.strip(),
    "birth_year": "",
})

# Loyalty records: split the name and the one-string address with named groups
names = loyal.member_name.apply(split_member_name)
addr = loyal.home_address.str.extract(
    r"^(?P<street>.+?),\s*(?P<city>[^,]+?),\s*(?P<state>[A-Z]{2})\s+(?P<zip>\d{5})$")
loyal_std = pd.DataFrame({
    "record_id": loyal.record_id,
    "first": names["first"].map(clean_name),
    "last": names["last"].map(clean_name),
    "email": loyal.email.map(clean_email),
    "phone": loyal.mobile.map(clean_phone),
    "street": addr.street.fillna("").map(clean_street),
    "city": addr.city.fillna("").str.strip().str.title(),
    "zip": addr.zip.fillna(""),
    "birth_year": loyal.birth_year,
})

# Stack both sources and add the canonical first name
people = pd.concat([web_std, loyal_std], ignore_index=True)
people["source"] = people.record_id.str[0].map({"W": "web", "L": "loyalty"})
people["first_canon"] = people["first"].map(lambda f: NICKNAMES.get(f, f))

# Checks
print("Rows:", len(people), "| web:", (people.source == "web").sum(), "| loyalty:", (people.source == "loyalty").sum())
print("Addresses that failed to split:", (people.street == "").sum())
print("Phones missing:", (people.phone == "").sum(), "| emails missing:", (people.email == "").sum())
print("Nicknames mapped:", (people["first"] != people.first_canon).sum())
display(people.sample(8, random_state=1))

Rows: 169 | web: 83 | loyalty: 86
Addresses that failed to split: 0
Phones missing: 17 | emails missing: 5
Nicknames mapped: 42


,record_id,first,last,email,phone,street,city,zip,birth_year,source,first_canon
48,W1048,jonny,berg,jonathan.berg30@outlook.com,6059369754,8146 S PHILLIPS AVE,Brookings,57050,,web,jonathan
90,L2007,ravi,mcallistre,ravi.mcallister77@outlook.com,6059087305,2536 S GRANGE AVE,Brandon,57047,2004,loyalty,ravi
19,W1019,mike,kowalczyk,michael.kowalczyk60@outlook.com,6055260269,1999 S LAKE AVE,Harrisburg,57098,,web,michael
35,W1035,nick,bradnt,nicholas.brandt89@outlook.com,,1151 S PHILLIPS AVE,Brandon,57079,,web,nicholas
127,L2044,deepa,mcallister,deepa.mcallister70@hotmail.com,6052481492,4757 S CLIFF AVE,Sioux Falls,57186,1966,loyalty,deepa
91,L2008,rebecca,fitzgerald,rebecca.fitzgerald56@outlook.com,6054800369,8814 S MAIN AVE,Sioux Falls,57131,1977,loyalty,rebecca
102,L2019,josephine,lindvqist,josephine.lindqvist@yahoo.com,6057044653,4607 S MINNESOTA AVE,Brandon,57051,,loyalty,josephine
14,W1014,patricia,nguyen,patricia.nguyen86@yahoo.com,,3619 S MINNESOTA AVE,Vermillion,57073,,web,patricia


**Nickname dictionary review.** The dictionary only includes nicknames that point to one formal name. Ambiguous entries a model might suggest were left out or checked against the data:

* "Sam" can be Samuel or Samantha, and "Alex" can be Alexander or Alexandra. Every Sam and Alex in this data is male, so mapping them to Samuel and Alexander is safe here.
* "Eliza" → Elizabeth was kept because the loyalty record for Eliza Hauge uses the email elizabeth.hauge@….

Last-name typos (e.g. "Nugyen") are deliberately not corrected in A1; the fuzzy name similarity in A2 handles them.

**Result:** 169 records (83 web + 86 loyalty), every loyalty address split into street, city and zip, and 42 nicknames mapped to formal names.

### A2. Blocking and scoring (12)
Generate candidate pairs with **at least two blocking keys** (for example phone, email, zip, or a last-name prefix plus
city), report how much blocking reduced the number of comparisons, compute similarity features, and combine them into a
score with two thresholds (match, gray zone, non-match). Justify your weights and thresholds using the dev pairs.

*Where this was taught:* Session 4, 0.5 (blocking on two keys and taking the union of the candidate pairs), 0.4 (which
similarity measure suits which field), and 3.1 (weights, thresholds, and precision and recall per policy).

In [6]:
# A2
from rapidfuzz.distance import JaroWinkler
from rapidfuzz import fuzz
from itertools import combinations

# Index by record_id and add the name part of the email (letters only)
p = people.set_index("record_id")
p["email_user"] = p.email.str.split("@").str[0].str.replace(r"[^a-z]", "", regex=True)

# Blocking: every pair of records that shares a key value
def block_pairs(key):
    pairs = set()
    for _, ids in p[p[key] != ""].groupby(key).groups.items():
        pairs.update(combinations(sorted(ids), 2))
    return pairs

blocks = {k: block_pairs(k) for k in ["phone", "email", "zip"]}
candidates = blocks["phone"] | blocks["email"] | blocks["zip"]

# How much blocking reduced the comparisons
n = len(p)
all_pairs = n * (n - 1) // 2
print("All possible pairs:", all_pairs)
for k, v in blocks.items():
    print(f"  pairs from {k} block: {len(v)}")
print(f"Candidate pairs (union): {len(candidates)}  ->  {1 - len(candidates) / all_pairs:.1%} fewer comparisons")

# Blocking recall: every dev match must survive blocking
dev_keys = [tuple(sorted(t)) for t in zip(dev_pairs.record_id_a, dev_pairs.record_id_b)]
dev_match_keys = [k for k, m in zip(dev_keys, dev_pairs.is_match) if m == 1]
print("Dev matches lost by blocking:", sum(k not in candidates for k in dev_match_keys), "of", len(dev_match_keys))

# Similarity features (Jaro-Winkler for short names, token sort for addresses, exact for IDs)
def features(a, b):
    x, y = p.loc[a], p.loc[b]
    return {
        "first_sim": JaroWinkler.similarity(x.first_canon, y.first_canon),
        "last_sim": JaroWinkler.similarity(x["last"], y["last"]),
        "email_same": float(x.email != "" and x.email == y.email),
        "email_user_sim": fuzz.ratio(x.email_user, y.email_user) / 100 if x.email_user and y.email_user else 0.0,
        "phone_same": float(x.phone != "" and x.phone == y.phone),
        "street_sim": fuzz.token_sort_ratio(x.street, y.street) / 100,
        "zip_same": float(x.zip == y.zip),
    }

# Weights: names carry the most because family members share phone, address and zip
WEIGHTS = {"first_sim": 0.25, "last_sim": 0.20, "email_same": 0.15, "email_user_sim": 0.10,
           "phone_same": 0.10, "street_sim": 0.15, "zip_same": 0.05}
MATCH_T, GRAY_T = 0.85, 0.55

def score_pairs(pairs):
    df = pd.DataFrame(sorted(pairs), columns=["a", "b"])
    df = pd.concat([df, pd.DataFrame([features(a, b) for a, b in zip(df.a, df.b)])], axis=1)
    df["score"] = df[list(WEIGHTS)].mul(pd.Series(WEIGHTS)).sum(axis=1).round(3)
    df["decision"] = np.where(df.score >= MATCH_T, "match", np.where(df.score >= GRAY_T, "gray", "non-match"))
    return df

# Score every candidate pair
scored = score_pairs(candidates)
print("\nDecisions on candidate pairs:")
print(scored.decision.value_counts().to_string())

# Score the dev pairs (pairs not in a block are automatic non-matches)
dev = dev_pairs.copy()
dev["key"] = dev_keys
dev_scores = score_pairs(set(dev_keys)).set_index(["a", "b"])
dev["score"] = [dev_scores.loc[k, "score"] if k in candidates else 0.0 for k in dev.key]
dev["decision"] = np.where(dev.score >= MATCH_T, "match", np.where(dev.score >= GRAY_T, "gray", "non-match"))
print("\nDev pairs: score range by true label")
print(dev.groupby("is_match").score.agg(["min", "max", "count"]))
print("\nDev pairs: decision by true label")
print(pd.crosstab(dev.decision, dev.is_match))

# Precision and recall on dev pairs under two policies
def pr(pred, truth):
    tp = int(((pred == 1) & (truth == 1)).sum()); fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else 0; rec = tp / (tp + fn) if tp + fn else 0
    return round(prec, 3), round(rec, 3)
print("\nPolicy 1 (gray = non-match): precision, recall =", pr((dev.decision == "match").astype(int), dev.is_match))
print("Policy 2 (gray = match):     precision, recall =", pr((dev.decision != "non-match").astype(int), dev.is_match))

# Dev pairs in the gray zone (these are why the LLM is needed)
display(dev[dev.decision == "gray"].sort_values("score")[["record_id_a", "record_id_b", "is_match", "score"]])

All possible pairs: 14196
  pairs from phone block: 55
  pairs from email block: 35
  pairs from zip block: 136
Candidate pairs (union): 136  ->  99.0% fewer comparisons
Dev matches lost by blocking: 0 of 20

Decisions on candidate pairs:
decision
non-match    71
match        39
gray         26

Dev pairs: score range by true label
           min    max  count
is_match                    
0         0.00  0.743     20
1         0.75  1.000     20

Dev pairs: decision by true label
is_match    0   1
decision         
gray        2   7
match       0  13
non-match  18   0

Policy 1 (gray = non-match): precision, recall = (1.0, 0.65)
Policy 2 (gray = match):     precision, recall = (0.909, 1.0)


,record_id_a,record_id_b,is_match,score
31,L2070,W1063,0,0.691
19,W1026,W1024,0,0.743
24,W1069,L2038,1,0.750
26,W1063,L2051,1,0.750
21,W1048,L2065,1,0.750
17,W1005,W1062,1,0.806
29,W1043,W1045,1,0.809
3,W1025,W1065,1,0.816
15,W1000,L2071,1,0.840


**Justification.** Blocking on phone, email and zip (union) cuts 14,196 possible pairs to 136 (99.0% fewer) and keeps all 20 dev matches. Names get the largest weights (first 0.25, last 0.20) because the hardest non-matches are family members who share the same phone, address and zip; only the name tells them apart. Jaro-Winkler suits short names with typos ("Nugyen"/"Nguyen"), and token-sort ratio suits addresses with reordered or abbreviated words. Shared email (0.15) is strong evidence, but people use different work and personal emails, so the email username similarity (0.10) adds partial credit. Phone, street and zip get smaller weights because families share them.

On the dev pairs, every score at or above **0.85** is a true match (precision 1.0), and every score below **0.55** is a true non-match. The scores between those two thresholds include true matches (0.75–0.84) and hard non-matches (the McAllister family at 0.69, two Oscar Lindqvists at the same address at 0.74), so no single cut-off separates them. Those 26 gray pairs go to the LLM in A3. Treating gray as non-match gives recall 0.65; treating gray as match gives precision 0.909, so neither alone is good enough.

### A3. LLM adjudication of the gray zone (12)
Send only gray-zone pairs to the model with a `same_person`, `confidence`, `reason` contract. Your instructions are your
matching policy: state how to treat nicknames, typos, work vs personal emails, and **family members** at the same
address (spouses, parent and child with the same name). Report the number of LLM calls.

In [7]:
# A3
from pydantic import BaseModel, Field

# The output contract for each gray-zone pair
class MatchDecision(BaseModel):
    same_person: bool
    confidence: float = Field(ge=0, le=1)
    reason: str

# Matching policy given to the model
POLICY = """You decide whether two customer records from Bluestem Coffee Co. (a web store and a loyalty
program) describe the SAME PERSON. Merging two different people is worse than missing a match, so when the
evidence is mixed, answer same_person = false.

Rules:
1. Nicknames are the same first name: Bill/Will = William, Kate/Kathy = Katherine, Jo/Josie = Josephine,
   Jonny/Jon = Jonathan, Mike = Michael, Bob/Rob/Bobby = Robert, Meg/Maggie/Peggy = Margaret, Pat/Trish = Patricia,
   Becky = Rebecca, Vicky = Victoria, Eliza/Liz = Elizabeth, Tommy = Thomas, Jen = Jennifer, Ben = Benjamin, etc.
2. Small typos in names are the same name (swapped or missing letters: "Nugyen" = "Nguyen", "Walhs" = "Walsh").
3. Different emails do NOT show different people: people use work emails (e.g. @avera.org, @citi.com,
   @state.sd.us, @usd.edu) and personal emails (gmail, yahoo, outlook, icloud, hotmail). A missing email or
   phone is not evidence either way.
4. FAMILY MEMBERS share an address and often a home phone. Two records at the same address with DIFFERENT first
   names (after nicknames) are different people (spouses, siblings, parent and child): same_person = false.
5. Same full name at the same address is usually one person, BUT if the two records have different phone numbers
   AND different personal emails, treat them as a parent and child with the same name: same_person = false.
   Different birth years (when both are given) also mean different people.
6. Different last names and different street addresses mean different people, even if the first names match.
Give confidence between 0 and 1 and a one-sentence reason."""

# Fields the model sees for each record (no internal scores)
SHOW = ["first", "last", "email", "phone", "street", "city", "zip", "birth_year", "source"]

def record_card(rid):
    r = p.loc[rid]
    return {"record_id": rid, **{k: r[k] for k in SHOW}}

# Send only the gray-zone pairs to the model
gray = scored[scored.decision == "gray"].copy()
print("Gray-zone pairs sent to the LLM:", len(gray))
calls_before = len(llm.calls)
results = []
for a, b in zip(gray.a, gray.b):
    content = {"record_A": record_card(a), "record_B": record_card(b)}
    try:
        d = llm.extract(content, MatchDecision, instructions=POLICY, temperature=0)
        results.append({"same_person": d.same_person, "confidence": d.confidence, "reason": d.reason})
    except Exception as exc:
        # If the model fails, keep the pair as a non-match (the safer choice)
        results.append({"same_person": False, "confidence": 0.0, "reason": f"LLM error: {type(exc).__name__}"})
gray[["same_person", "confidence", "reason"]] = pd.DataFrame(results, index=gray.index)

# Accept an LLM match only when it is reasonably confident
gray["llm_match"] = gray.same_person & (gray.confidence >= 0.6)

# Report LLM calls and usage
print("LLM calls made:", len(llm.calls) - calls_before, "(includes cached calls and repair retries)")
print(llm.usage())

# Show every gray-zone decision with both names
names = p["first"] + " " + p["last"]
gray["name_a"], gray["name_b"] = gray.a.map(names), gray.b.map(names)
pd.set_option("display.max_colwidth", 120)
display(gray[["a", "name_a", "b", "name_b", "score", "same_person", "confidence", "llm_match", "reason"]])

# Final pair decisions: automatic matches plus LLM-approved gray pairs
scored["final_match"] = (scored.decision == "match")
scored.loc[gray.index, "final_match"] = gray.llm_match
print("Final matched pairs:", int(scored.final_match.sum()),
      "| automatic:", int((scored.decision == "match").sum()), "| from LLM:", int(gray.llm_match.sum()))

Gray-zone pairs sent to the LLM: 26
[llmkit] ServerError (attempt 1); waiting 2s before retrying.
[llmkit] ClientError (attempt 1); waiting 2s before retrying.
[llmkit] ClientError (attempt 1); waiting 2s before retrying.
[llmkit] ClientError (attempt 2); waiting 4s before retrying.
[llmkit] ClientError (attempt 1); waiting 2s before retrying.
[llmkit] ClientError (attempt 2); waiting 4s before retrying.
[llmkit] ClientError (attempt 1); waiting 2s before retrying.
LLM calls made: 26 (includes cached calls and repair retries)
calls                               26
cached_calls                         0
input_tokens                     21488
output_tokens                     1506
avg_latency_s                     0.91
est_cost_usd     set price_per_million
Name: google/gemini-3.5-flash-lite, dtype: object


,a,name_a,b,name_b,score,same_person,confidence,llm_match,reason
4,L2002,leila peterson,W1001,leila peterosn,0.845,True,0.95,True,"The records share the exact same first name, phone number, and street address, with only a minor typo in the last na..."
20,L2010,william novka,W1058,will novak,0.841,True,0.95,True,"The records share the same phone number, exact same address, matching first name (Will/William), and a minor surname..."
25,L2012,patricia berg,W1045,pat berg,0.809,True,0.95,True,"Pat is a recognized nickname for Patricia, and both records share the exact same last name, phone number, and street..."
29,L2016,oscar lindqvist,W1026,oscar lindqvist,0.750,False,0.85,False,The records share the same name and address but have completely different phone numbers and different personal email...
50,L2033,jonny abdi,W1071,jon abdi,0.750,True,0.95,True,"The records share the same first name variant (Jonny/Jon for Jonathan), last name, phone number, and street address."
52,L2035,paige nguyen,W1075,paige nguyen,0.750,True,1.00,True,"The records share the exact same first name, last name, phone number, and street address."
56,L2037,josephine tran,W1062,jo tran,0.806,True,0.95,True,"Jo is a recognized nickname for Josephine, and both records share the exact same last name, phone number, and street..."
57,L2038,nadia gonzalez,W1069,nadia gonzalez,0.750,True,0.95,True,"Both records share the exact same first name, last name, street address, city, and zip code, and the use of differen..."
68,L2045,trish brandt,L2057,patricia berg,0.568,False,1.00,False,"The records have different last names and different street addresses, indicating they are different people."
75,L2051,fatima mcallister,L2070,erin mcallister,0.618,False,0.99,False,The two records share an address and phone number but have different first names (Fatima and Erin) and different bir...


Final matched pairs: 60 | automatic: 39 | from LLM: 21


**A3 summary.** Only the 26 gray-zone pairs were sent to the model (Google gemini-3.5-flash-lite), so A3 used **26 LLM calls** (about 21,500 input and 1,500 output tokens). Seven calls hit temporary rate-limit or server errors and were retried automatically; none failed.

The model judged **21 pairs as the same person and 5 as different people**:

* Fatima vs Erin McAllister (2 pairs): family members at the same address and phone.
* The two Oscar Lindqvists (2 pairs): same name and address, but different phone and personal email, so treated as parent and child.
* Trish Brandt vs Patricia Berg: different last name and different address.

A match was accepted only when `same_person` was true with confidence of at least 0.6. Final matched pairs: 60 (39 automatic + 21 from the LLM).

### A4. Clusters and evaluation (11)
Turn matched pairs into clusters (connected components), assign a `cluster_id` to every record, and evaluate pairwise
precision, recall, and F1 on the dev pairs. Inspect every dev error and explain it.

*Where this was taught:* Session 4, 3.3 (union-find, and `cluster_pair_metrics`); a dev pair counts as a predicted
match when both of its records have the same `cluster_id` (Session 4, 0.6).

In [8]:
# A4
# Union-find: link every matched pair, then each root is one customer
parent = {rid: rid for rid in people.record_id}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[max(rx, ry)] = min(rx, ry)

matched = scored[scored.final_match]
for a, b in zip(matched.a, matched.b):
    union(a, b)

# Assign a cluster_id to every record (C001, C002, ...)
roots = people.record_id.map(find)
root_to_cid = {r: f"C{i:03d}" for i, r in enumerate(sorted(roots.unique()), start=1)}
clusters = pd.DataFrame({"record_id": people.record_id, "cluster_id": roots.map(root_to_cid)})
people["cluster_id"] = clusters.cluster_id.values
sizes = clusters.cluster_id.value_counts()
print("Records:", len(clusters), "| clusters (customers):", clusters.cluster_id.nunique())
print("Cluster sizes:", sizes.value_counts().sort_index().to_dict())

# Dev evaluation: a pair is predicted a match when both records share a cluster_id
cid = clusters.set_index("record_id").cluster_id
dev["pred"] = (dev.record_id_a.map(cid) == dev.record_id_b.map(cid)).astype(int)
tp = int(((dev.pred == 1) & (dev.is_match == 1)).sum())
fp = int(((dev.pred == 1) & (dev.is_match == 0)).sum())
fn = int(((dev.pred == 0) & (dev.is_match == 1)).sum())
precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
print(f"\nDev pairs: TP={tp} FP={fp} FN={fn}")
print(f"Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")

# Inspect every dev error
errors = dev[dev.pred != dev.is_match]
print("\nDev errors:", len(errors))
if len(errors):
    display(errors[["record_id_a", "record_id_b", "is_match", "pred", "score", "decision"]])

# Check the hard dev non-matches ended up in different clusters
hard = dev[(dev.is_match == 0) & (dev.decision == "gray")]
display(hard.assign(cluster_a=hard.record_id_a.map(cid), cluster_b=hard.record_id_b.map(cid))
        [["record_id_a", "record_id_b", "cluster_a", "cluster_b", "pred"]])

# Show the multi-record clusters so they can be checked by eye
multi = people[people.cluster_id.map(sizes) > 1].sort_values(["cluster_id", "record_id"])
display(multi[["cluster_id", "record_id", "first", "last", "email", "phone", "street"]])

Records: 169 | clusters (customers): 113
Cluster sizes: {1: 61, 2: 48, 3: 4}

Dev pairs: TP=20 FP=0 FN=0
Precision=1.000  Recall=1.000  F1=1.000

Dev errors: 0


,record_id_a,record_id_b,cluster_a,cluster_b,pred
19,W1026,W1024,C095,C017,0
31,L2070,W1063,C071,C052,0


,cluster_id,record_id,first,last,email,phone,street
85,C003,L2002,leila,peterson,leila.peterson@avera.org,6056006820,7820 S CLIFF AVE APT 27
1,C003,W1001,leila,peterosn,leila.peterson9@gmail.com,6056006820,7820 S CLIFF AVE APT 27
86,C004,L2003,deepa,christensen,deepa.christensen@gmail.com,6058386013,5580 S MINNESOTA AVE
55,C004,W1055,deepa,christensne,deepa.christensen@gmail.com,6058386013,5580 S MINNESOTA AVE
87,C005,L2004,eliza,hauge,elizabeth.hauge@outlook.com,6054198799,9463 S MINNESOTA AVE
...,...,...,...,...,...,...,...
53,C086,W1053,jo,abdi,josephine.abdi@yahoo.com,6056074040,7267 S WESTERN AVE APT 5
22,C093,W1022,elizabeth,sorensen,esorensen@hotmail.com,6053985119,1401 S PHILLIPS AVE
81,C093,W1081,elizabeth,soernsen,elizabeth.sorensen84@icloud.com,6053985119,1401 S PHILLIPS AVE
30,C097,W1030,jennifer,okafor,jokafor@gmail.com,6057061900,1764 S CLIFF AVE


**Dev evaluation.** Pairwise precision, recall and F1 on the 40 dev pairs are all **1.000** (TP = 20, FP = 0, FN = 0), so there are no dev errors to explain. The two places errors were most likely were checked directly:

1. **Hard non-matches in the gray zone.** Erin vs Fatima McAllister (L2070/W1063, same address and phone) and the two Oscar Lindqvists (W1024/W1026, same name and address) landed in different clusters. The LLM policy treats different first names at one address as family members, and the same name with a different phone and email as a parent and child.
2. **Matches created by clustering.** Clustering can link records that were never scored directly (if A = B and B = C, then A = C). The four three-record clusters (Patricia/Pat Berg, Josephine/Josie/Jo Tran, Thomas/Tommy Okafor, Kai Walsh) were checked by eye: in each, all records share the same phone and address and have the same first name after nickname mapping.

A perfect score on 40 dev pairs does not guarantee the same on the hidden labels: a different family pair could still be merged, which is the main remaining risk.

---
# Part B. Golden records (15 points)
Create one golden record per cluster. Write your **survivorship rules** in a markdown cell first (which first name, which
email, which phone, which address, and why), then implement them. Report how many customers are web-only, loyalty-only,
and in both programs (Session 4, 3.3 counts the source systems in each cluster the same way).


**Survivorship rules:** one golden record per cluster, built field by field.

* **First name:** the formal name after nickname mapping (`first_canon`), so Jo/Josie/Josephine becomes "Josephine". If records disagree, the most common one wins, then the most recent record. A formal name is the safest choice for mailings and matches future records best.
* **Last name:** the spelling that also appears in one of the customer's email usernames (e.g. "Goznalez" becomes "Gonzalez" because the email is heidi.gonzalez82@…). Typos come from staff typing names in, but customers type their own email, so the email is the more reliable spelling. If no email confirms a spelling, the most common one wins, then the most recent record.
* **Email:** a personal email (gmail, yahoo, outlook, icloud, hotmail) before a work email (avera.org, citi.com, state.sd.us, usd.edu, sanfordhealth.org), because people lose work email when they change jobs and some employers block marketing mail. Among personal emails, the most recent record wins.
* **Phone:** the most common non-empty phone in the cluster, then the most recent record. A phone repeated across systems has been confirmed more than once.
* **Address:** street, city and zip are kept together from one record (never mixed across records), choosing the most common full address, then the most recent record, because people move and the newest record is most likely current.
* **Birth year:** only the loyalty program collects it, so it comes from the loyalty record when there is one.
* **Programs:** a customer is "both" if the cluster has at least one web and one loyalty record, otherwise "web only" or "loyalty only".

**Result:** 113 customers: **50 in both programs, 36 loyalty-only, 27 web-only.**

In [9]:
# Part B
# Record dates (needed for "most recent" tie-breaks)
dates = pd.concat([web.set_index("record_id").created_at, loyal.set_index("record_id").member_since])
people["record_date"] = people.record_id.map(dates)

WORK_DOMAINS = {"avera.org", "citi.com", "state.sd.us", "usd.edu", "sanfordhealth.org"}

# Most frequent non-empty value; ties go to the value from the most recent record
def most_common(values, order):
    df = pd.DataFrame({"v": values, "o": order})
    df = df[df.v != ""]
    if df.empty:
        return ""
    counts = df.v.value_counts()
    top = counts[counts == counts.max()].index
    return df[df.v.isin(top)].sort_values("o").v.iloc[-1]

# Last name: a spelling confirmed by an email username wins (fixes typos such as "Goznalez")
def pick_last(g):
    tokens = [t for e in g.email if e for t in re.sub(r"\d", "", e.split("@")[0]).split(".")]
    for l in g["last"]:
        if l and l.replace(" ", "") in tokens:
            return l
    for l in g["last"]:
        close = [t for t in tokens if l and JaroWinkler.similarity(t, l.replace(" ", "")) >= 0.85]
        if close:
            fixed = close[0]
            if " " in l:   # put the space back ("runninghorse" -> "running horse")
                cut = l.index(" ")
                fixed = fixed[:cut] + " " + fixed[cut:]
            return fixed
    return most_common(g["last"], g.record_date)

# Title case that keeps "McAllister"
def nice_case(name):
    return " ".join(("Mc" + w[2:].title()) if w.lower().startswith("mc") else w.title() for w in name.split())

# Email: personal email first (people leave jobs), then the most recent record
def pick_email(g):
    e = g[g.email != ""].copy()
    if e.empty:
        return ""
    e["personal"] = ~e.email.str.split("@").str[1].isin(WORK_DOMAINS)
    return e.sort_values(["personal", "record_date"]).email.iloc[-1]

# Address: most common full address kept together; ties go to the most recent record
def pick_address(g):
    addr = g.street + "|" + g.city + "|" + g.zip
    return most_common(addr, g.record_date).split("|")

# Build one golden record from a cluster's records
def golden(g):
    street, city, zip_ = pick_address(g)
    return pd.Series({
        "first_name": most_common(g.first_canon, g.record_date).title(),
        "last_name": nice_case(pick_last(g)),
        "email": pick_email(g),
        "phone": most_common(g.phone, g.record_date),
        "street": street.title(), "city": city, "zip": zip_,
        "birth_year": most_common(g.birth_year, g.record_date),
        "n_records": len(g),
        "record_ids": ", ".join(sorted(g.record_id)),
        "in_web": (g.source == "web").any(),
        "in_loyalty": (g.source == "loyalty").any(),
    })

# One golden record per cluster
golden_records = people.drop(columns="cluster_id").groupby(people["cluster_id"]).apply(golden).reset_index()
golden_records["program"] = np.select(
    [golden_records.in_web & golden_records.in_loyalty, golden_records.in_web],
    ["both", "web only"], default="loyalty only")

# Program counts
print("Golden records (customers):", len(golden_records))
print(golden_records.program.value_counts().to_string())

# Show the merged customers so the rules can be checked
display(golden_records[golden_records.n_records > 1].drop(columns=["in_web", "in_loyalty"]))

Golden records (customers): 113
program
both            50
loyalty only    36
web only        27


,cluster_id,first_name,last_name,email,phone,street,city,zip,birth_year,n_records,record_ids,program
2,C003,Leila,Peterson,leila.peterson9@gmail.com,6056006820,7820 S Cliff Ave Apt 27,Sioux Falls,57196,,2,"L2002, W1001",both
3,C004,Deepa,Christensen,deepa.christensen@gmail.com,6058386013,5580 S Minnesota Ave,Sioux Falls,57113,1987,2,"L2003, W1055",both
4,C005,Elizabeth,Hauge,elizabeth.hauge@outlook.com,6054198799,9463 S Minnesota Ave,Vermillion,57064,1973,2,"L2004, W1051",both
6,C007,Robert,Larsen,robert.larsen56@gmail.com,6056321895,9681 S Louise Ave Apt 17,Vermillion,57036,1982,2,"L2006, W1027",both
9,C010,Wyatt,McAllister,wyatt.mcallister5@yahoo.com,6052351757,3539 S Louise Ave,Brandon,57047,1996,2,"L2009, W1036",both
10,C011,William,Novak,william.novak71@yahoo.com,6056393618,6162 S Minnesota Ave Apt 16,Harrisburg,57047,1989,2,"L2010, W1058",both
11,C012,Oscar,Kowalczyk,oscar.kowalczyk@icloud.com,6054867035,9498 S Phillips Ave Apt 25,Brookings,57064,1958,2,"L2011, W1007",both
12,C013,Patricia,Berg,pberg@gmail.com,6059434945,3373 S Minnesota Ave,Tea,57039,2001,3,"L2012, W1043, W1045",both
15,C016,Heidi,Novak,heidi.novak@icloud.com,6054661927,6369 S Kiwanis Ave,Brookings,57003,1999,2,"L2015, W1038",both
16,C017,Oscar,Lindqvist,oscar.lindqvist52@gmail.com,6057281652,9093 S Summit Ave,Brookings,57028,,2,"L2016, W1024",both


---
# Part C. An LLM-written log parser (25 points)
`pos_register.log` mixes three formats. Following Session 4 (Approach C):
1. Give the model 6 to 8 sample lines (from different formats) and the target fields: `timestamp` (`YYYY-MM-DD HH:MM:SS`,
   clock time exactly as written: **do not** convert time zones), `level` (INFO, WARN, ERROR; the one-letter codes are
   I, W, E), `store_id`, `employee_id`, `event` (upper case with underscores, e.g. `DRAWER_OPEN`), and `amount` (dollars
   with two decimals, empty if none; JSON amounts are in cents). Ask for a standard-library-only function `parse_line`.
2. **Read the code before running it.** Then test it on the 15 dev lines, feed failures back to the model, and iterate.
   Keep every version and its dev accuracy in a small table.
3. Parse all 60 lines. Compare the cost of this approach with sending every line to the model.

*Where this was taught:* Session 4, 1.3: the generation prompt, reading and testing the code, and the repair loop that
sends failing lines back (`failing_lines`). Note one difference: Session 4's target asked for Central time, so it
converted UTC; this target asks for the clock time **as written**, so do not convert. Always read the target schema.

In [11]:
# Part C: prompts, generated code (read it!), tests, iterations


# Part C, step 1: ask the model to write the parser
SAMPLE_LINES = [log_lines[i - 1] for i in (1, 2, 4, 8, 9, 24, 32)]   # all three formats, AM and PM, with and without amounts

GEN_PROMPT = f"""Write a Python function `parse_line(line: str) -> dict` that parses one line of a point-of-sale
register log. The log mixes three formats. Here are sample lines:

{chr(10).join(SAMPLE_LINES)}

Return a dict with exactly these keys, all values as strings:
- "timestamp": "YYYY-MM-DD HH:MM:SS", 24-hour clock. Keep the clock time EXACTLY as written in the line.
  Do NOT convert time zones (a trailing "Z" is ignored, not converted). Convert AM/PM to 24-hour
  (12:30:15 PM -> 12:30:15, 01:02:58 PM -> 13:02:58, 12:05:00 AM -> 00:05:00).
- "level": one of "INFO", "WARN", "ERROR". One-letter codes map I -> INFO, W -> WARN, E -> ERROR;
  words such as "info"/"warn"/"error" are upper-cased.
- "store_id": e.g. "VRM1".
- "employee_id": e.g. "E116".
- "event": upper case with underscores, e.g. "drawer open" -> "DRAWER_OPEN", "refund" -> "REFUND".
- "amount": dollars with exactly two decimals, e.g. "37.11", "7.50". Empty string "" when the line has no amount.
  JSON amounts ("amt_cents") are in cents: 750 -> "7.50", 2220 -> "22.20". "$37.11" -> "37.11".

Rules: use only the Python standard library (re, json, datetime). Do not read files. Do not print.
Return ONLY the code in one ```python block, with no explanation."""

# Take the code out of a ```python ... ``` block if the model used one
def extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

calls_before_c = len(llm.calls)
code_v1 = extract_code(llm.ask(GEN_PROMPT, temperature=0))
versions = [{"version": "v1", "code": code_v1}]

# READ THIS CODE before running the next cell
print(code_v1)


from datetime import datetime
import json
import re


def parse_line(line: str) -> dict:
    line = line.strip()

    # Determine format
    if line.startswith("{"):
        data = json.loads(line)
        # ISO timestamp format: "2026-02-14T06:11:43Z"
        ts_raw = data.get("ts", "")
        # Remove trailing Z if present and replace T with space
        ts_clean = ts_raw.rstrip("Z").replace("T", " ")
        if "." in ts_clean:
            ts_clean = ts_clean.split(".")[0]

        timestamp = ts_clean
        level = data.get("severity", "").upper()
        store_id = data.get("loc", "")
        employee_id = data.get("user", "")
        event_raw = data.get("type", "")
        event = event_raw.upper().replace(" ", "_")

        amount = ""
        if "amt_cents" in data:
            amt_cents = int(data["amt_cents"])
            amount = f"{amt_cents / 100:.2f}"
        elif "amount" in data:
            amount = f"{float(data['amount']):.2f}"

    elif "[" in line and "]" in l

**Code review (v1, read before running).** The function picks the format from the first character: `{` means JSON, `[LEVEL]` means key=value, and anything else is the pipe format. Checks made before running:

1. **Safe to run:** it imports only `datetime`, `json` and `re`, and it does not read files, use the network or call `eval`.
2. **No time zone conversion:** JSON timestamps only drop the `Z` and the `T`, so the clock time is kept exactly as written, as the target requires (unlike Session 4, which converted UTC to Central time).
3. **AM/PM handled correctly:** the pipe format uses `strptime` with `%I ... %p`, so 12:30 PM stays 12:30 and 1:02 PM becomes 13:02.
4. **Cents converted:** `amt_cents` is divided by 100 and formatted with two decimals (750 → 7.50).
5. **Missing amounts handled:** the pipe parser finds the amount by the `$` sign rather than by position, so lines without an amount ("drawer open") still get the right employee ID and an empty amount.

One harmless leftover: the variable `pairs` is created but never used.

In [13]:
# Part C, step 2: test on the 15 dev lines, repair failures, keep every version
FIELDS = ["timestamp", "level", "store_id", "employee_id", "event", "amount"]
ALLOWED_IMPORTS = {"re", "json", "datetime"}

# Refuse code that imports anything outside the standard modules we allowed
def load_parser(code):
    imports = set(re.findall(r"^\s*(?:from|import)\s+(\w+)", code, re.M))
    if not imports <= ALLOWED_IMPORTS:
        raise ValueError(f"disallowed imports: {imports - ALLOWED_IMPORTS}")
    namespace = {}
    exec(code, namespace)
    return namespace["parse_line"]

# Run the parser on one line; a crash becomes an "ERROR" value instead of stopping the cell
def run_parser(parse_line, line):
    try:
        out = parse_line(line)
        return {f: ("" if out.get(f) is None else str(out.get(f))) for f in FIELDS}
    except Exception as exc:
        return {f: f"ERROR: {type(exc).__name__}" for f in FIELDS}

# Compare every field of every dev line with the gold answer
def test_on_dev(code):
    parse_line = load_parser(code)
    failing = []
    for _, gold in log_dev.iterrows():
        line = log_lines[gold.line_no - 1]
        got = run_parser(parse_line, line)
        wrong = {f: (got[f], gold[f]) for f in FIELDS if got[f] != gold[f]}
        if wrong:
            failing.append({"line_no": gold.line_no, "line": line, "wrong": wrong})
    accuracy = 1 - len(failing) / len(log_dev)
    return accuracy, failing

# Repair prompt: send the failing lines (got vs expected) back to the model
def repair_prompt(code, failing):
    lines = "\n".join(
        f"LINE: {f['line']}\n  " + "; ".join(f"{k}: got {g!r}, expected {e!r}" for k, (g, e) in f["wrong"].items())
        for f in failing)
    return (f"Your parse_line function gave wrong output on these lines:\n\n{lines}\n\n"
            f"Here is your current code:\n```python\n{code}\n```\n\n"
            "Fix the code so these lines are parsed correctly without breaking other lines. Keep the same rules "
            "(standard library only, keep clock time as written, no time zone conversion). "
            "Return ONLY the full corrected code in one ```python block.")

# Test v1, then repair up to 3 times while any dev line fails
MAX_REPAIRS = 3
code = versions[0]["code"]
for round_no in range(MAX_REPAIRS + 1):
    try:
        acc, failing = test_on_dev(code)
    except Exception as exc:
        acc, failing = 0.0, [{"line_no": 0, "line": "(code did not load)", "wrong": {"load": (str(exc), "")}}]
    versions[round_no]["dev_accuracy"] = acc
    versions[round_no]["failing_lines"] = [f["line_no"] for f in failing]
    print(f"{versions[round_no]['version']}: dev accuracy {acc:.0%}, failing lines {versions[round_no]['failing_lines']}")
    for f in failing:
        print("   ", f["line_no"], f["wrong"])
    if not failing or round_no == MAX_REPAIRS:
        break
    code = extract_code(llm.ask(repair_prompt(code, failing), temperature=0))
    versions.append({"version": f"v{round_no + 2}", "code": code})
    print(f"\n--- {versions[-1]['version']} code (read it) ---\n{code}\n")

# Version table
version_table = pd.DataFrame(versions)[["version", "dev_accuracy", "failing_lines"]]
display(version_table)

# Parse all 60 lines with the final version
final_parser = load_parser(versions[-1]["code"])
parsed_log = pd.DataFrame([{"line_no": i, **run_parser(final_parser, line)} for i, line in enumerate(log_lines, start=1)])

# Sanity checks on all 60 lines (the gold file only covers 15)
checks = {
    "timestamp format": parsed_log.timestamp.str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"),
    "level valid": parsed_log.level.isin(["INFO", "WARN", "ERROR"]),
    "store_id format": parsed_log.store_id.str.fullmatch(r"[A-Z]+\d+"),
    "employee_id format": parsed_log.employee_id.str.fullmatch(r"E\d+"),
    "event format": parsed_log.event.str.fullmatch(r"[A-Z]+(_[A-Z]+)*"),
    "amount format": parsed_log.amount.str.fullmatch(r"(\d+\.\d{2})?"),
}
print("Lines passing each check (of 60):", {k: int(v.sum()) for k, v in checks.items()})
print("Event counts:", parsed_log.event.value_counts().to_dict())
display(parsed_log)

# Cost: generating a parser vs sending every line to the model
c_calls = llm.calls[calls_before_c:]
gen_in = sum(c["input_tokens"] for c in c_calls); gen_out = sum(c["output_tokens"] for c in c_calls)
per_line_in = estimate_tokens(GEN_PROMPT) + 150   # instructions + schema
per_line_tokens = sum(per_line_in + estimate_tokens(l) + 60 for l in log_lines)   # ~60 output tokens each
print(f"\nCode-generation approach: {len(c_calls)} LLM calls, {gen_in + gen_out:,} tokens in total, "
      "then the 60 lines (or 60 million) are parsed for free.")
print(f"One-call-per-line approach (estimate): 60 LLM calls, about {per_line_tokens:,} tokens, "
      f"and about {per_line_tokens / 60 * 1_000_000:,.0f} tokens for 1 million lines.")

v1: dev accuracy 100%, failing lines []


,version,dev_accuracy,failing_lines
0,v1,1.0,[]


Lines passing each check (of 60): {'timestamp format': 60, 'level valid': 60, 'store_id format': 60, 'employee_id format': 60, 'event format': 60, 'amount format': 60}
Event counts: {'SALE': 29, 'LOGIN': 10, 'REFUND': 6, 'SYNC_FAIL': 5, 'DRAWER_OPEN': 5, 'VOID': 5}


,line_no,timestamp,level,store_id,employee_id,event,amount
0,1,2026-02-14 06:01:16,WARN,VRM1,E116,REFUND,37.11
1,2,2026-02-14 06:11:43,INFO,VRM1,E100,LOGIN,
2,3,2026-02-14 06:27:06,ERROR,SF01,E127,SYNC_FAIL,
3,4,2026-02-14 06:42:54,ERROR,VRM1,E128,SYNC_FAIL,
4,5,2026-02-14 06:44:08,INFO,BRK1,E108,SALE,28.44
5,6,2026-02-14 06:47:39,WARN,SF02,E133,REFUND,42.77
6,7,2026-02-14 06:57:38,INFO,SF01,E138,DRAWER_OPEN,
7,8,2026-02-14 07:10:28,INFO,BRK1,E116,SALE,22.20
8,9,2026-02-14 07:31:56,INFO,SF02,E120,SALE,28.74
9,10,2026-02-14 07:46:00,INFO,SF02,E154,LOGIN,



Code-generation approach: 1 LLM calls, 2,045 tokens in total, then the 60 lines (or 60 million) are parsed for free.
One-call-per-line approach (estimate): 60 LLM calls, about 39,762 tokens, and about 662,700,000 tokens for 1 million lines.


**Iterations.**

| Version | Dev accuracy (15 lines) | Failing lines |
|---|---|---|
| v1 | 100% | none |

Version 1 scored 100% on the 15 dev lines, so the repair loop (which sends failing lines back to the model with the expected and actual values) had nothing to fix and no further versions were needed. Because the dev file covers only 15 lines, all 60 parsed lines were also checked against format rules (timestamp pattern, valid level, ID patterns, event in UPPER_SNAKE case, amount with two decimals or empty), and **all 60 passed**. Spot checks on the tricky lines outside the dev set were also correct: line 32 (12:30 PM → 12:30:15), line 35 (01:02 PM → 13:02:58), line 21 (750 cents → 7.50) and line 24 ("drawer open" → DRAWER_OPEN, no amount).

A strong first version came from a precise prompt: it named the traps directly (no time zone conversion, 12 PM handling, cents, missing amounts) and included sample lines covering all three formats.

**Cost comparison.**

| Approach | LLM calls | Tokens (this log, 60 lines) | Tokens for 1 million lines |
|---|---|---|---|
| Generate a parser once (this approach) | 1 | 2,045 | 2,045 (no extra calls) |
| Send every line to the model | 60 | about 39,762 | about 663 million |

Generating the parser took **1 LLM call and 2,045 tokens**. After that, the code parses the 60 lines, or 60 million, at no extra LLM cost and in milliseconds.

Sending every line to the model would take **60 calls and about 39,762 tokens** (about 19 times more) for this log alone, and about **663 million tokens for 1 million lines**, because the instructions are re-sent with every line.

Per-line LLM parsing also adds latency, rate-limit waits, and a small chance of a different answer on every line. The generated code is deterministic and can be tested.

**Conclusion:** the LLM is worth its cost for writing and repairing the parser, not for running it.

---
# Part D. Deliverables and reflection (15 points)
1. Save `hw4_clusters.csv` (`record_id, cluster_id` for **all 169 records**) and `hw4_parsed_log.csv`
   (`line_no, timestamp, level, store_id, employee_id, event, amount` for all 60 lines).
2. **Reflection (250 words):** For a loyalty program, which is worse: merging two different people or failing to merge
   one person's records? How did that judgment shape your thresholds and LLM instructions? What did the gray zone cost
   in LLM calls, and how would the cost scale to 2 million customers?

In [16]:
# Part D: save files

# Cluster file: record_id and cluster_id for all 169 records
clusters_out = people[["record_id", "cluster_id"]]
clusters_out.to_csv("hw4_clusters.csv", index=False)

# Parsed log file: all 60 lines with the required columns
log_out = parsed_log[["line_no", "timestamp", "level", "store_id", "employee_id", "event", "amount"]]
log_out.to_csv("hw4_parsed_log.csv", index=False)

# Check the saved files
check_c = pd.read_csv("hw4_clusters.csv")
check_l = pd.read_csv("hw4_parsed_log.csv")
print("hw4_clusters.csv:", check_c.shape, "| unique records:", check_c.record_id.nunique(), "| clusters:", check_c.cluster_id.nunique())
print("hw4_parsed_log.csv:", check_l.shape)

# Numbers for the reflection
gray_calls = len(gray)
print(f"Gray-zone LLM calls: {gray_calls} of {len(scored)} candidate pairs; "
      f"{gray_calls / len(people):.3f} calls per record")

# Download both files to your computer
from google.colab import files
files.download("hw4_clusters.csv")
files.download("hw4_parsed_log.csv")

hw4_clusters.csv: (169, 2) | unique records: 169 | clusters: 113
hw4_parsed_log.csv: (60, 7)
Gray-zone LLM calls: 26 of 136 candidate pairs; 0.154 calls per record


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


**Reflection:**

For a loyalty program, **merging two different people is worse than failing to merge one person's records.** A false merge puts one person's points, purchase history and contact details on another person's account: a parent could redeem a child's points, offers would go to the wrong email, and correcting it requires untangling transactions by hand while exposing private data. A missed merge costs less: one customer has two accounts and marketing counts them twice, which is annoying but easy to fix later and harms no one else.

**How this shaped my thresholds and LLM instructions.** The automatic match threshold (0.85) sits above every dev non-match, so automatic merges had precision 1.0. Names carry the largest weights because family members share a phone, address and zip. The LLM instructions say that mixed evidence means "not the same person", that different first names at one address are family members, and that the same name with a different phone and a different personal email is a parent and child. I also accepted an LLM match only at confidence 0.6 or higher. The result kept the McAllister family and the two Oscar Lindqvists apart, with a dev F1 of 1.0.

**Cost of the gray zone and scaling.** The gray zone cost **26 LLM calls** (about 23,000 tokens) out of 136 candidate pairs, or about 0.15 calls per record. At that rate, 2 million customers would need about **300,000 calls and roughly 270 million tokens**. That is affordable with a low-cost model, but blocking on zip alone would create far too many candidate pairs at that size. I would need tighter blocking keys (zip plus a last-name prefix), response caching, and batch requests to keep the gray zone small.

---
# Make your submission PDF (run this last)
Save your work first (**File > Save**), then run the cell below. It turns the **whole** notebook, outputs included,
into `HW4_submission.pdf` and downloads it, plus an HTML copy as a backup. The first run takes one to two minutes because it
installs a PDF printer.

Do not use **File > Print** in Colab: it prints only what is laid out on screen, so long notebooks are cut off partway
down a page. If the PDF step fails, open the downloaded `HW4_submission.html` in your browser and print that page to PDF
(Ctrl+P or Cmd+P, then *Save as PDF*); a standalone page prints in full.

In [17]:
#@title Make the PDF and HTML copies of this notebook
SUBMISSION_NAME = "HW4_submission"     # file names of the PDF and HTML copies
UPLOAD_INSTEAD = False      # True: convert an uploaded .ipynb instead of this notebook

import json, os, subprocess, sys, textwrap

def _notebook_json():
    """The current notebook, outputs included. Colab first; otherwise ask for the downloaded .ipynb."""
    global SUBMISSION_NAME
    try:
        if UPLOAD_INSTEAD:
            raise LookupError("upload requested")
        from google.colab import _message
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
        if nb and nb.get("cells"):
            return nb
        print("Colab did not return the notebook; falling back to an upload.")
    except ImportError:
        pass
    except LookupError:
        pass
    except Exception as exc:
        print(f"Could not read the notebook from Colab ({type(exc).__name__}); falling back to an upload.")
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab: in Jupyter, use File > Save and Export Notebook As > PDF or HTML instead.")
        return None
    if UPLOAD_INSTEAD:
        print("Choose the .ipynb file you downloaded from your assignment (File > Download > Download .ipynb).")
    else:
        print("Download this notebook (File > Download > Download .ipynb), then choose that file below.")
    uploaded = files.upload()
    if not uploaded:
        return None
    name, data = next(iter(uploaded.items()))
    if UPLOAD_INSTEAD:
        SUBMISSION_NAME = os.path.splitext(os.path.basename(name))[0].replace(" ", "_")
    return json.loads(data.decode("utf-8"))

PRINT_CSS = """
<style>
  @page { size: Letter; margin: 0.5in; }
  body { background: white !important; }
  .jp-OutputArea-output, .jp-Cell-outputArea, .output_scroll, .jp-OutputArea { overflow: visible !important; max-height: none !important; }
  pre, code { white-space: pre-wrap !important; word-break: break-word; }
  .jp-RenderedHTMLCommon table { font-size: 10px; table-layout: auto; }
  .jp-RenderedHTMLCommon td, .jp-RenderedHTMLCommon th { word-break: break-word; }
  img { max-width: 100% !important; height: auto !important; }
  .jp-Cell { break-inside: auto; }
</style>
"""

def _to_html(nb_dict, path):
    try:
        import nbformat
        from nbconvert import HTMLExporter
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbconvert"], check=True)
        import nbformat
        from nbconvert import HTMLExporter
    nb_dict.setdefault("nbformat", 4); nb_dict.setdefault("nbformat_minor", 5)
    node = nbformat.reads(json.dumps(nb_dict), as_version=4)   # joins sources and outputs stored as lists of lines
    for cell in node.cells:
        if cell.get("cell_type") == "code" and cell.get("metadata", {}).get("cellView") == "form":
            first = cell["source"].split("\n", 1)[0]   # course-provided form cells (setup, data): title only
            cell["source"] = (first if first.startswith("#@title") else "#@title (setup cell)") + "\n# code hidden in this copy"
    node.metadata.pop("widgets", None)            # widget state makes the page heavy and is not needed on paper
    html, _ = HTMLExporter(template_name="lab").from_notebook_node(node)
    html = html.replace("</head>", PRINT_CSS + "</head>", 1)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return path

PRINTER = textwrap.dedent("""
    import sys
    from playwright.sync_api import sync_playwright
    src, out = sys.argv[1], sys.argv[2]
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page()
        page.goto("file://" + src, wait_until="load", timeout=120000)
        try:
            page.wait_for_load_state("networkidle", timeout=20000)   # lets MathJax finish if a cell uses math
        except Exception:
            pass
        page.emulate_media(media="screen")
        page.pdf(path=out, format="Letter", print_background=True, scale=0.85,
                 margin=dict(top="0.5in", bottom="0.5in", left="0.5in", right="0.5in"))
        browser.close()
""")

def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)

def _to_pdf(html_path, pdf_path):
    try:
        import playwright  # noqa: F401
    except ImportError:
        print("Installing the PDF printer (one time per Colab session)...")
        _run([sys.executable, "-m", "pip", "install", "-q", "playwright"])
    _run([sys.executable, "-m", "playwright", "install", "chromium"])
    with open("_print_pdf.py", "w") as f:
        f.write(PRINTER)
    cmd = [sys.executable, "_print_pdf.py", os.path.abspath(html_path), os.path.abspath(pdf_path)]
    r = _run(cmd)
    if r.returncode != 0:                         # usually missing system libraries on a fresh machine
        _run([sys.executable, "-m", "playwright", "install-deps", "chromium"])
        r = _run(cmd)
    if r.returncode != 0:
        print("PDF step failed:", (r.stderr or r.stdout).strip().splitlines()[-1:])
        return None
    return pdf_path

nb_json = _notebook_json()
if nb_json is not None:
    n_cells = len(nb_json.get("cells", []))
    n_out = sum(1 for c in nb_json.get("cells", []) if c.get("cell_type") == "code" and c.get("outputs"))
    print(f"Notebook read: {n_cells} cells, {n_out} with outputs.")
    if n_out == 0:
        print("Warning: no outputs found. Run your cells (Runtime > Run all) and save before exporting.")
    html_file = _to_html(nb_json, SUBMISSION_NAME + ".html")
    pdf_file = _to_pdf(html_file, SUBMISSION_NAME + ".pdf")
    try:
        from google.colab import files
        for path in [pdf_file, html_file]:
            if path:
                files.download(path)
    except ImportError:
        pass
    if pdf_file:
        print(f"Done: {pdf_file} ({os.path.getsize(pdf_file) // 1024} KB) and {html_file}. "
              "Open the PDF and check the last page before you submit.")
    else:
        print(f"Only the HTML copy was made: open {html_file} in your browser and print it to PDF (Ctrl+P or Cmd+P).")

Notebook read: 32 cells, 12 with outputs.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done: HW4_submission.pdf (184 KB) and HW4_submission.html. Open the PDF and check the last page before you submit.
